In [ ]:
import os
import numpy as np
import cv2
from PIL import Image
import torch
import albumentations as A
from torch.utils.data import DataLoader
from torchvision.transforms import functional as F
from typing import List, Dict, Tuple, Union, Final

## Configurations
### Model parameters

In [ ]:
# number of classes (including background)
# the order of objects when creating the semnatic masks is important for semantic segmentation
# we create the semnatic masks in this order: bg, cage, cell, and then bead
# as cells can be inside cages (creating holes in cage masks), and beads
# can potentially be over the cells (creating holes)
LABEL_MAP: Dict[int, str] = {1: 'cage', 2: 'cell', 3: 'bead'}
NUM_CLASSES: Final[int] = len(LABEL_MAP) + 1

MODEL_PATH = 'checkpoints'
if not os.path.exists(MODEL_PATH):
    os.mkdir(MODEL_PATH)

# model input image large/small-side sizes
MODEL_INPUT_SIZE: Final[int] = 518

TRANSFORM_MEAN: Final[np.ndarray] = np.array([0.485, 0.456, 0.406])
TRANSFORM_STD: Final[np.ndarray] = np.array([0.229, 0.224, 0.225])

### Dataset parameters
Use the script in `Semantic Segmentation over Cage Crops.ipynb` to create the cropped images for cages. This step should be repeated for any additional dataset/cell type available. Point to the location of these cropped imaged below. 

In [ ]:
CROPPED_IMAGES_FOLDER = 'cage_crops_data'

## Data Model
### Dataset class

In [ ]:
from sem_seg_utils import SemanticMaskDataset

### Image and segmentation mask transforms
Here, we use albumentations package that takes both image and annotations to apply the transformations on them. It takes and returns numpy arrays. 

In [ ]:
train_transform = A.Compose([A.HorizontalFlip(), 
                             A.VerticalFlip(), 
                             A.GridDistortion(p=0.2), 
                             A.RandomBrightnessContrast(brightness_limit = (-0.2, 0.2), 
                                                        contrast_limit = (-0.2, 0.2)),
                             A.GaussNoise()])

### Datasets and Dataloaders

In [ ]:
# datasets
train_dataset = SemanticMaskDataset(images_path=os.path.join(CROPPED_IMAGES_FOLDER, "images", "train"), 
                                    masks_path=os.path.join(CROPPED_IMAGES_FOLDER, "masks", "train"),
                                    mean=TRANSFORM_MEAN, std=TRANSFORM_STD, 
                                    model_input_size= (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE), 
                                    transform=train_transform)

test_dataset = SemanticMaskDataset(images_path=os.path.join(CROPPED_IMAGES_FOLDER, "images", "test"), 
                                   masks_path=os.path.join(CROPPED_IMAGES_FOLDER, "masks", "test"), 
                                   mean=TRANSFORM_MEAN, std=TRANSFORM_STD, 
                                   model_input_size= (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE), )

## Visualization

In [ ]:
from sem_seg_utils import to_numpy, show_sample

In [ ]:
img = show_sample(31, train_dataset)
Image.fromarray(img[:, :, ::-1])

## Model Definition

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

class Dinov2ModelSemanticSegmentation(torch.nn.Module):
    def __init__(self, num_classes: int, model_input_size: int, model_type: str = "small", with_registers: bool=True) -> None:
        super().__init__()
        # get the pre-trained DINOv2 model based on the passed size for the model
        model_type_map = {"small": "dinov2_vits14", 
                          "base":  "dinov2_vitb14", 
                          "large": "dinov2_vitl14", 
                          "giant": "dinov2_vitg14",
                         }
        if model_type in model_type_map:
            model_type_str: str = model_type_map[model_type]
                
        else:
            model_type_str: str = "dinov2_vitb14"
            print(f"[ERROR] Incorrect model type passed {model_type}! Using the base model by default.")
        
        # DINOv2 with registers
        if with_registers:
            model_type_str += "_reg"

        # the backbone of the Semantic segmentation model 
        # we will add a linear classifier head on top of the patch embeddings
        self.dinov2_backbone = torch.hub.load("facebookresearch/dinov2", model_type_str)
        # the
        self.patch_size: int = self.dinov2_backbone.patch_size
        
        if model_input_size % self.patch_size != 0:
            print(f"[ERROR] The model input size {model_input_size} should be a mutiple of the DINOv2 model's patch size {self.patch_size}!")
            print("Re-instantiate the class with a correct model_input_size!")
        
        # freeze the model
        for param in self.dinov2_backbone.parameters():
            param.requires_grad_(False)

        # number of classes to predict
        self.num_classes: int = num_classes
        # the dimension of Dinov2 embeddings
        self.dinov2_embeddings_dim: int = self.dinov2_backbone.embed_dim
        # the number of patches in the image will be self.num_patches_per_img_side ** 2
        self.num_patches_per_img_side: int = int(model_input_size / self.patch_size)
        # the classifier head (linear)
        self.classifier = torch.nn.Conv2d(self.dinov2_embeddings_dim, self.num_classes, kernel_size=(1, 1), stride=(1, 1)) 

    def forward(self, img_tensor: torch.Tensor) -> torch.Tensor:
        # patch_embeddings will be of size num_batches x  self.num_patches_per_img_side ** 2 X  self.dinov2_embeddings_dim
        patch_embeddings: torch.tensor = self.dinov2_backbone.forward_features(img_tensor)['x_norm_patchtokens']
        # reshape the embeddings before applying the linear head 
        patch_embeddings = patch_embeddings.reshape(-1, 
                                                    self.num_patches_per_img_side, 
                                                    self.num_patches_per_img_side, 
                                                    self.dinov2_embeddings_dim)
        patch_embeddings = patch_embeddings.permute(0, 3, 1, 2)
        # logits will of size num_batches x self.num_classes x self.num_patches_per_img_side x self.num_patches_per_img_side
        # after the linear operation
        logits: torch.tensor = self.classifier(patch_embeddings)
        # extrapolate the pixels to the original size of the input image
        results = {}
        results['out'] = torch.nn.functional.interpolate(logits, size=img_tensor.shape[2:], mode="bilinear", align_corners=False)
        return results


In [ ]:
model = Dinov2ModelSemanticSegmentation(
    num_classes=NUM_CLASSES, 
    model_input_size=MODEL_INPUT_SIZE, 
    model_type="large", 
    with_registers=False
)
model.to(device)

## Training
### Training parameters

In [ ]:
# training batch size
# this should be at least 2 as the DeepLab model Batch norm require at least 2
BATCH_SIZE: Final[int] = 16
# learning rate
LEARNING_RATE: Final[float] = 1e-3
# number of training epochs
NUM_EPOCHS = 6
# learning rate decay steps, a value of 0 means One-cycle LR scheduler should be used
LR_DECAY_STEPS = 0

### Dataloaders

In [ ]:
# drop_last is set to True to avoid passing a data with batch size of 1
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

### Optimizer, LR scheduler and loss function

In [ ]:
# construct an optimizer
# Adam optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr = LEARNING_RATE)

print(f"Adam Optimizer is configured for {NUM_EPOCHS} epochs")

print(f"Initial learning rate is set to {LEARNING_RATE}")
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} with {len(train_loader)} steps per epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_loader))
    
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                   step_size=LR_DECAY_STEPS,
                                                   gamma=0.1)



criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

### Performance metrics

In [ ]:
from sem_seg_utils import mIoU, pixel_accuracy

### Training script

In [ ]:
from sem_seg_utils import train

In [ ]:
history  = train(model,
                 NUM_CLASSES,
                 train_loader, 
                 test_loader, 
                 False, # do not ignore the bg index  
                 optimizer, 
                 lr_scheduler,
                 NUM_EPOCHS,
                 device, 
                 MODEL_PATH
                )

### Saving the best/final model with some model configurations

In [ ]:
BEST_CHECKPOINT = 'checkpoint_5.pt'
model = Dinov2ModelSemanticSegmentation(
    num_classes=NUM_CLASSES, 
    model_input_size=MODEL_INPUT_SIZE, 
    model_type="large", 
    with_registers=False
)
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, BEST_CHECKPOINT)))
model_param_dict = {}
model_param_dict['model_state_dict'] = model.state_dict()
model_param_dict['label_map'] = LABEL_MAP
model_param_dict['input_size'] = MODEL_INPUT_SIZE
torch.save(model_param_dict, os.path.join(MODEL_PATH, 'final.pt'))

## Testing

In [ ]:
NUM_CLASSES: Final[int] = 4
LABEL_MAP: Dict[int, str] = {1: 'cage', 2: 'cell', 3: 'bead'}

MODEL_PATH = 'checkpoints'
CHECKPOINT_NAME = 'checkpoint_5.pt'
# model input image large/small-side sizes
MODEL_INPUT_SIZE: Final[int] = 518


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model = seg_models.deeplabv3_resnet50(weights=seg_models.DeepLabV3_ResNet50_Weights.DEFAULT)
model = Dinov2ModelSemanticSegmentation(num_classes=NUM_CLASSES, 
                                        model_input_size=MODEL_INPUT_SIZE, 
                                        model_type="large", 
                                        with_registers=False)
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, CHECKPOINT_NAME)))

In [ ]:
from sem_seg_utils import evaluate

In [ ]:
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'final.pt'))['model_state_dict'])
test_iou_score, test_accuracy, test_loss = evaluate(model, NUM_CLASSES, test_loader, device, return_loss=True, ignore_index_zero=False)
print("Test loss: {:.3f}".format(test_loss))
print("Test mean IoU: {:.3f}".format(test_iou_score))
print("Test Accuracy: {:.3f}".format(test_accuracy))

In [ ]:
def predict(model, input_image, device):
    
    model.eval()
    model.to(device)

    input_shape = input_image.shape
    # save the dimensions for returning the mask with the same ones
    org_img_height, org_img_width = input_shape[:2]
    
    # the model expect images in 3-channel RGB format
    if len(input_shape) < 3:
        image = cv2.cvtColor(input_image, cv2.COLOR_GRAY2RGB)
    else:
        image = input_image

    # resize to model input size
    image = cv2.resize(image, dsize=(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE))
    # normalize
    image = (image.astype(float) / 255.0 - TRANSFORM_MEAN) / TRANSFORM_STD
    
    image_tensor = F.to_tensor(image).unsqueeze(dim=0).to(device).float()
    
    with torch.no_grad():
        output = model(image_tensor)["out"]
        mask = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)
        
    return cv2.resize(mask, dsize=(org_img_width, org_img_height), interpolation=cv2.INTER_NEAREST)

def predict_batch(model, input_images_list, device):
    
    model.eval()
    model.to(device)

    # convert to 3-channel images if needed, and store the original image dimensions for 
    # post processing
    images_tensor_list: List[torch.tensor] = []
    org_img_dims: List[Tuple[int, int]] = []
    
    for img in input_images_list:
        img_shape: tuple = img.shape
        if len(img_shape) < 3:
            image = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            image = img
        # resize to model input size
        image = cv2.resize(image, dsize=(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE))
        # normalize
        image = (image.astype(float) / 255.0 - TRANSFORM_MEAN) / TRANSFORM_STD
        
        images_tensor_list.append(F.to_tensor(image).to(device).float())
        org_img_dims.append(img_shape[:2])

    images_tensors = torch.stack(images_tensor_list, dim=0)
    with torch.no_grad():
        outputs = model(images_tensors)["out"]
        masks = torch.argmax(outputs, dim=1).squeeze().cpu().numpy().astype(np.uint8)
    
    return [cv2.resize(mask, (org_img_dims[i][1], org_img_dims[i][0]), interpolation=cv2.INTER_NEAREST) for i, mask in enumerate(masks)]

In [ ]:
idx = 21932
img_t, mask_t = test_dataset[idx]
image: np.ndarray = to_numpy(img_t.permute(1, 2, 0).squeeze())
mask_gt: np.ndarray = to_numpy(mask_t).astype(np.uint8)
# scale back and add the mean, scale to 0-255
image = ((image * train_dataset.std + train_dataset.mean) * 255).mean(axis=2).astype(np.uint8)
mask = predict(model, image, device)

In [ ]:
Image.fromarray(show_sample(idx, test_dataset))

In [ ]:
Image.fromarray(mask * 63)

In [ ]:
import time
start = time.time()
for i in range(100):
    mask = predict(model, image, device)
print(f"Dinov2 with linear head inference took {np.round((time.time() - start) * 10, 2)} ms")

In [ ]:
batch_size = 4
start = time.time()
for i in range(100):
    masks = predict_batch(model, [image] * batch_size, device)
print(f"Dinov2 with linear head inference took {np.round((time.time() - start) * 10, 2)} ms per batch of size {batch_size}")
print(f"Dinov2 with linear head inference took {np.round((time.time() - start) * 10 / batch_size, 2)} ms per image")